# COMM117 Week 7 Workshop 

**How to use:** Try the **TODO** cells first. Each TODO is followed by a **Model answer** cell.

**This workshop covers **
- LLM API mental model 
- Prompt structure template (system/user/constraints/output/guardrails)
- Text generation/decoding: greedy, beam search, sampling, temperature, top‑k, top‑p



## Part 0 — Setup (do not edit)

In [ ]:
# Part 0 — Setup (do not edit)

import math
import numpy as np
from typing import List, Dict, Any


## Part 1 — LLM APIs

### Exercise 1.1 —API Call

Generate a 1–2 sentence explanation of KV cache.


In [ ]:
# Model answer — Exercise 1.1 (Gemini API call)
# (Assumes `client` is already created from earlier setup in Colab)
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain KV cache in at most two sentences. No bullet points."
)
print(response.text)


### Exercise 1.2 — Prompt structure template (write a good prompt)

Use Gemini API to extract structured information from a paragraph.

## Summarise the vLLM paper
"https://arxiv.org/pdf/2309.06180"



Write a prompt (as Python strings) that follows this structure:
- **System:** role + rules
- **User:** task + inputs
- **Constraints:** length/style
- **Output format:** JSON schema
- **Guardrail:** if missing info, ask ONE question

Task: *Extract `{topic, key_claim, evidence}` from a paragraph.*


In [ ]:
from google.genai import types

PAPER_LINK = "https://arxiv.org/pdf/2309.06180"

# Model answer — Exercise 1.2 (prompt structure)
SYSTEM = (
    "You are a careful research summarisation assistant. "
    "Output JSON only (no markdown, no extra text). "
    "Do not invent results; if unsure, use null."
)

USER = (
    f"Summarise the paper at: {PAPER_LINK}\n"
    "Focus on: (1) serving bottleneck, (2) PagedAttention idea, "
    "(3) what Figure 3 vs Figure 5 mean, (4) what experiments support performance."
)

CONSTRAINTS = (
    "Write concise, factual answers. Keep one_sentence_summary <= 25 words. "
    "Use only information available to you; if you cannot be sure, set fields to null."
)

OUTPUT_FORMAT = (
    "Return exactly this JSON:\n"
    "{\n"
    "  \"one_sentence_summary\": string,\n"
    "  \"problem\": string,\n"
    "  \"core_idea\": string,\n"
    "  \"fig3_interpretation\": string|null,\n"
    "  \"fig5_interpretation\": string|null,\n"
    "  \"performance_evidence\": [string],\n"
    "  \"limitations_or_assumptions\": string|null,\n"
    "  \"needs_clarification\": boolean,\n"
    "  \"question\": string|null\n"
    "}"
)

GUARDRAIL = (
    "If you cannot access enough paper content to answer faithfully, "
    "set needs_clarification=true and ask EXACTLY ONE question in 'question' "
    "(e.g., ask for the abstract or key figures)."
)

{
  "system": SYSTEM,
  "user": USER,
  "constraints": CONSTRAINTS,
  "output_format": OUTPUT_FORMAT,
  "guardrail": GUARDRAIL,
}


## Part 2 — Text generation (decoding & sampling)
### Exercise 2.1 —  Compare outputs under different sampling settings

Run the **same** `SYSTEM` + `prompt` with four configs:

1) low temperature (0.2)  
2) high temperature (1.0)  
3) stricter nucleus sampling (top_p=0.6, temp=0.8)  
4) looser nucleus sampling (top_p=0.95, temp=0.8)

Then answer:
- Which config is best for **faithful summarisation**?
- Which config produces the most **paraphrasing**?
- Which config is most likely to break the **JSON-only** requirement?


In [ ]:
from google.genai import types

# Model answer — Exercise 2.1 (call Gemini with the prompt structure)
prompt = "\n\n".join([USER, CONSTRAINTS, OUTPUT_FORMAT, GUARDRAIL])

resp = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        types.Content(role="system", parts=[types.Part(text=SYSTEM)]),
        types.Content(role="user", parts=[types.Part(text=prompt)]),
    ],
    config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=450),
)

print(resp.text)


### Excercise 2.2 - Load GPT-2 (tokenizer + model) in Hugging Face.

- Use the prompt: “Explain KV cache in one sentence:”

Generate outputs using:

1) Greedy (no sampling)

2) Sampling with temperature=1.0

3) Top-k sampling (top_k=50, temperature=0.8)

4) Top-p sampling (top_p=0.9, temperature=0.8)

- Run the sampling methods twice each and print outputs.

-  which setting is most consistent vs most diverse?

In [ ]:
# OPTIONAL (run if you want)
# !pip -q install -U transformers torch

from transformers import AutoTokenizer, AutoModelForCausalLM

name = "gpt2"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)
model.eval()

prompt = "Explain KV cache in one sentence:"
inputs = tok(prompt, return_tensors="pt")

# Greedy
out_greedy = model.generate(**inputs, max_new_tokens=40, do_sample=False)
print("GREEDY:\n", tok.decode(out_greedy[0], skip_special_tokens=True), "\n")

# Sampling: top-p
out_topp = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
)
print("TOP-P:\n", tok.decode(out_topp[0], skip_special_tokens=True), "\n")

# Sampling: top-k
out_topk = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.8,
    top_k=50,
)
print("TOP-K:\n", tok.decode(out_topk[0], skip_special_tokens=True))


In [ ]:
# Model answer — short comparison (Exercise 2.2)
comparison = (
    "Greedy is the most consistent (deterministic). "
    "Top-p and top-k are more diverse because they sample tokens; "
    "top-p is adaptive to the probability mass, while top-k fixes the candidate set size."
)
print(comparison)
